# CodonTransformer csi_top10_hc biological evaluation

Read-only evaluation of the 594-record independent test split: real CDS versus the official pretrained baseline versus the best formal fine-tuned checkpoint. This notebook does not train a model. Prediction caches and reports are written to Google Drive so an interrupted free Colab session can resume.

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), "CUDA GPU is required"
print(torch.cuda.get_device_name(0))

## Clone the private project securely and pin upstream
Store a fine-grained read-only token as the Colab secret `GITHUB_TOKEN`. The token is passed only through `GIT_ASKPASS`.

In [ ]:
import json
import os
import subprocess
import tempfile
from pathlib import Path
from google.colab import userdata

REPO_URL = "https://github.com/Yuano0o/codontransformer-nb.git"
PROJECT_DIR = Path("/content/codontransformer-project")
UPSTREAM_URL = "https://github.com/Adibvafa/CodonTransformer.git"
UPSTREAM_COMMIT = "4a447b01dab860feb81b647ff1ff88ad598517f4"
HF_MODEL_ID = "adibvafa/CodonTransformer"
HF_MODEL_REVISION = "9744dcc920d813066391fc828d7a590207f148e8"
github_token = userdata.get("GITHUB_TOKEN")
if not github_token:
    raise RuntimeError("Colab secret GITHUB_TOKEN is unavailable")
with tempfile.TemporaryDirectory(prefix="git-askpass-") as askpass_dir:
    askpass = Path(askpass_dir) / "askpass.sh"
    askpass.write_text(
        '#!/bin/sh\ncase "$1" in\n'
        '  *sername*) printf \'%s\\n\' \'x-access-token\' ;;\n'
        '  *assword*) printf \'%s\\n\' "$COLAB_GITHUB_TOKEN" ;;\n'
        '  *) exit 1 ;;\nesac\n', encoding="utf-8")
    askpass.chmod(0o700)
    env = os.environ.copy()
    env.update({"GIT_ASKPASS": str(askpass), "GIT_TERMINAL_PROMPT": "0", "COLAB_GITHUB_TOKEN": github_token})
    if not (PROJECT_DIR / ".git").is_dir():
        subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True, env=env)
    else:
        subprocess.run(["git", "-C", str(PROJECT_DIR), "pull", "--ff-only"], check=True, env=env)
    env.pop("COLAB_GITHUB_TOKEN", None)
github_token = None
os.chdir(PROJECT_DIR)
UPSTREAM_DIR = PROJECT_DIR / "upstream"
if not (UPSTREAM_DIR / ".git").is_dir():
    subprocess.run(["git", "clone", UPSTREAM_URL, str(UPSTREAM_DIR)], check=True)
subprocess.run(["git", "-C", str(UPSTREAM_DIR), "checkout", UPSTREAM_COMMIT], check=True)

## Install evaluation dependencies and mount Drive

In [ ]:
subprocess.run([os.sys.executable, "-m", "pip", "install", "-r", "requirements-colab.txt"], check=True)
os.environ["PYTHONPATH"] = os.pathsep.join(filter(None, (str(UPSTREAM_DIR), os.environ.get("PYTHONPATH", ""))))
if str(UPSTREAM_DIR) not in os.sys.path:
    os.sys.path.insert(0, str(UPSTREAM_DIR))
from google.colab import drive
drive.mount("/content/drive")

## Verify immutable inputs
Upload the local `data/processed/n_benthamiana/stage2/codon_reference.json` once to `MyDrive/CodonTransformer/data/stage2/codon_reference.json`. The existing test JSONL and best checkpoint are reused read-only.

In [ ]:
DRIVE_ROOT = Path("/content/drive/MyDrive/CodonTransformer")
TEST_PATH = DRIVE_ROOT / "data/stage2/final_csi_cohorts/experiments/csi_top10_hc/test.jsonl"
REFERENCE_PATH = DRIVE_ROOT / "data/stage2/codon_reference.json"
RUN_DIR = DRIVE_ROOT / "runs/finetune_csi_top10_hc_formal_v1"
CHECKPOINT = RUN_DIR / "checkpoints/best-epoch04-val_loss0.842573.ckpt"
OUTPUT_DIR = RUN_DIR / "biological_evaluation_v1"
for label, path in (("test", TEST_PATH), ("reference", REFERENCE_PATH), ("checkpoint", CHECKPOINT)):
    assert path.is_file(), f"Missing {label}: {path}"
with TEST_PATH.open(encoding="utf-8") as handle:
    assert sum(bool(line.strip()) for line in handle) == 594
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Persistent evaluation output:", OUTPUT_DIR)

## Download the exact official pretrained baseline

In [ ]:
MODEL_DIR = PROJECT_DIR / "models/pretrained"
subprocess.run([
    os.sys.executable, "scripts/download_pretrained.py",
    "--repo-id", HF_MODEL_ID, "--revision", HF_MODEL_REVISION,
    "--output-dir", str(MODEL_DIR),
], check=True)
assert (MODEL_DIR / "model.safetensors").is_file()

## Generate/resume predictions and run paired biological statistics
The two deterministic prediction caches are saved every 25 records. Re-running this cell resumes missing predictions and does not train either model.

In [ ]:
subprocess.run([
    os.sys.executable, "scripts/evaluate_biological_fidelity.py",
    "--model-dir", str(MODEL_DIR),
    "--checkpoint", str(CHECKPOINT),
    "--test-dataset", str(TEST_PATH),
    "--reference-json", str(REFERENCE_PATH),
    "--output-dir", str(OUTPUT_DIR),
    "--device", "cuda",
    "--organism", "Nicotiana tabacum",
    "--expected-records", "594",
    "--expected-test-sha256", "78c9c0d54ab20465b9cecd71d087169396baca09ce823b5443d5b3e07e8f7ec3",
    "--expected-reference-sha256", "3d63f51d45972365fc92e7e20052ac82453cb10b09fcf26b76e5f6aaf0eec908",
    "--expected-pretrained-sha256", "23994e1e7324e78c9d9da71040e8677c0a635287cf1c7c9ea5096b7eadc44dfd",
    "--bootstrap-samples", "10000",
    "--seed", "23",
    "--flush-every", "25",
], check=True)

## Inspect the decision and persistent artifacts

In [ ]:
summary = json.loads((OUTPUT_DIR / "biological_evaluation_summary.json").read_text())
print(json.dumps(summary["decision"], indent=2))
print((OUTPUT_DIR / "biological_evaluation_report.md").read_text())
for path in sorted(OUTPUT_DIR.rglob("*")):
    if path.is_file():
        print(path.relative_to(OUTPUT_DIR), path.stat().st_size)